In [1]:
import estrategias.mean_reversion_light as mrl
import estrategias.mean_reversion_super_light as mrsl
import pandas as pd
import numpy as np
import preprocessamento as pp

import e3_screening as scr
from datetime import date


In [2]:
estrategias = [mrl]
data_base = date.today()
n_anos = 5
ticker_unico = 'MRVE3.SA'
modo_grade = False

In [3]:
def preparar_df(ticker: str, data_base: date) -> pd.DataFrame:
    df = pp.montar_dataset(ticker, dias_total_historico=600, data_base=data_base)
    df = pp.adicionar_features_sniper(df)

    # Retornos
    for n in [1, 3, 5, 10, 20]:
        col = f"ret_{n}d"
        if col not in df.columns:
            df[col] = df["close"].pct_change(n)

    # MMs do Ibovespa
    if "close_ibov" in df.columns:
        df["mm50_ibov"]  = df["close_ibov"].rolling(50).mean()
        df["mm100_ibov"] = df["close_ibov"].rolling(100).mean()
        df["mm200_ibov"] = df["close_ibov"].rolling(200).mean()

    # MMs do papel
    df["mm20"] = df["close"].rolling(20).mean()
    df["mm50"] = df["close"].rolling(50).mean()

    # Máximas de períodos (sem lookahead)
    for n in [10, 15, 20, 30]:
        df[f"max_{n}d"] = df["high"].shift(1).rolling(n).max()

    # ATR
    if "atr" not in df.columns and "ATR_14" in df.columns:
        df["atr"] = df["ATR_14"]
    elif "atr" not in df.columns:
        df["prev_close"] = df["close"].shift(1)
        df["tr"] = pd.concat([
            df["high"] - df["low"],
            (df["high"] - df["prev_close"]).abs(),
            (df["low"]  - df["prev_close"]).abs(),
        ], axis=1).max(axis=1)
        df["atr"] = df["tr"].rolling(14).mean()

    # ATR percentil (para estratégias que usam)
    def _percentil_rank(series, window=60):
        def _pct(x):
            if len(x) < 2 or pd.isna(x.iloc[-1]):
                return 50
            return (x < x.iloc[-1]).sum() / len(x) * 100
        return series.rolling(window).apply(_pct, raw=False)

    df["atr_percentil"] = _percentil_rank(df["atr"], window=60)

    # Swing lows (sem lookahead)
    for janela in [1, 2, 3, 5, 10]:
        min_n_ant = df["low"].shift(1).rolling(janela).min()
        cond = (
            (df["low"] <= min_n_ant) &
            (df["low"] <  df["low"].shift(1)) &
            (df["low"] <= df["low"].shift(2))
        )
        df[f"swing_low_{janela}"]       = cond
        df[f"swing_low_value_{janela}"] = df["low"].where(cond, np.nan).ffill()

    # Mínimas recentes
    for janela in [10, 20, 30]:
        df[f"min_{janela}d"] = df["low"].shift(1).rolling(janela).min()

    df["__ticker__"] = ticker
    return df

In [4]:
df = preparar_df(ticker_unico, date(2026, 12, 31))

[MRVE3.SA] Pós-Download Total: Nenhuma divergência significativa nas amostras (tolerância 1.00%).
✅ Usando cache local: /palpiteiro/dados\macro_20230715_20270101.csv
[^BVSP] Pós-Download Total: Nenhuma divergência significativa nas amostras (tolerância 1.00%).
[BOVA11.SA] Pós-Download Total: Nenhuma divergência significativa nas amostras (tolerância 1.00%).


In [5]:
df['ret_5d']

data
2024-07-15         NaN
2024-07-16         NaN
2024-07-17         NaN
2024-07-18         NaN
2024-07-19         NaN
                ...   
2026-03-09   -0.143856
2026-03-10   -0.086044
2026-03-11   -0.111340
2026-03-12   -0.130621
2026-03-13   -0.170968
Name: ret_5d, Length: 416, dtype: float64

In [6]:
p_mrl = mrl._parametros()
p_mrsl = mrsl._parametros()

In [7]:
df['sinal_mrl'] = df.apply(lambda r: mrl.fn_sinal(r, p_mrl), axis=1)
df['sinal_mrsl'] = df.apply(lambda r: mrsl.fn_sinal(r, p_mrsl), axis=1)

In [13]:
p_mrl

{'rsi_min': 20,
 'rsi_max': 40,
 'queda_5d_min': -0.03,
 'vol_relativo_min': 0.5,
 'adx_max': 40,
 'close_acima_mm50': True,
 'ibov_acima_mm200': True,
 'alvo_pct': 0.03,
 'stop_pct': 0.04,
 'max_dias': 15}

In [8]:
colunas = ['close','close_ibov','ret_5d','rsi', 'mm20','mm50', 'mm200_ibov', 'vol_relativo', 'ADX_14','sinal_mrl','sinal_mrsl']

In [9]:
df[colunas].tail(10)

,close,close_ibov,ret_5d,rsi,mm20,mm50,mm200_ibov,vol_relativo,ADX_14,sinal_mrl,sinal_mrsl
data,,,,,,,,,,,
2026-03-02,10.01,189307.0000,-0.018627,63.970482,9.5235,8.5488,150779.600000,0.937541,32.075928,False,False
2026-03-03,9.53,183105.0000,-0.079227,52.636136,9.5965,8.5726,150998.455000,1.361949,30.019350,False,False
2026-03-04,9.70,185366.0000,-0.061896,55.634306,9.6675,8.6064,151229.350000,0.667485,28.192904,False,False
2026-03-05,9.34,180464.0000,-0.104506,48.616089,9.7140,8.6324,151433.490000,0.914979,26.709541,False,False
2026-03-06,9.30,179365.0000,-0.091797,47.893153,9.7700,8.6604,151629.765000,1.143251,25.902788,False,False
2026-03-09,8.57,180915.0000,-0.143856,37.061574,9.7615,8.6756,151844.935000,2.206438,26.115409,False,False
2026-03-10,8.71,183447.0000,-0.086044,39.870230,9.7500,8.6976,152075.805000,1.916228,26.312843,True,False
2026-03-11,8.62,183969.0000,-0.111340,38.675372,9.7260,8.7152,152306.530000,1.152664,26.496174,False,False
2026-03-12,8.12,179285.0000,-0.130621,32.795212,9.6580,8.7238,152512.275000,1.517056,27.308156,False,False


In [10]:
df[colunas].tail()

,close,close_ibov,ret_5d,rsi,mm20,mm50,mm200_ibov,vol_relativo,ADX_14,sinal_mrl,sinal_mrsl
data,,,,,,,,,,,
2026-03-09,8.57,180915.0000,-0.143856,37.061574,9.7615,8.6756,151844.935000,2.206438,26.115409,False,False
2026-03-10,8.71,183447.0000,-0.086044,39.870230,9.7500,8.6976,152075.805000,1.916228,26.312843,True,False
2026-03-11,8.62,183969.0000,-0.111340,38.675372,9.7260,8.7152,152306.530000,1.152664,26.496174,False,False
2026-03-12,8.12,179285.0000,-0.130621,32.795212,9.6580,8.7238,152512.275000,1.517056,27.308156,False,False
2026-03-13,7.71,178103.8125,-0.170968,28.913261,9.5510,8.7242,152705.089062,1.887537,28.696142,False,False


In [11]:
def _val(row: pd.Series, col: str, default: float = np.nan) -> float:
    v = row.get(col, default)
    return default if pd.isna(v) else float(v)

def _retorno_nd(row: pd.Series, n: int) -> float:
    return _val(row, f"ret_{n}d", default=-999)

def fn_sinal(row: pd.Series, p: dict) -> bool:
    #debug de cada condicao
    print(f'RSI >= rsi_min: {_val(row, "rsi")} >= {p["rsi_min"]} -> {_val(row, "rsi") >= p["rsi_min"]}')
    print(f'RSI <= rsi_max: {_val(row, "rsi")} <= {p["rsi_max"]} -> {_val(row, "rsi") <= p["rsi_max"]}')
    print(f'Retorno 5d <= queda_5d_min: {_retorno_nd(row, 5)} <= {p["queda_5d_min"]} -> {_retorno_nd(row, 5) <= p["queda_5d_min"]}')
    print(f'Close > MM20: {_val(row, "close")} > {_val(row, "mm20", 0)} -> {_val(row, "close") > _val(row, "mm20", 0)}')
    print(f'Close Ibov > MM200 Ibov: {_val(row, "close_ibov")} > {_val(row, "mm200_ibov", 0)} -> {_val(row, "close_ibov") > _val(row, "mm200_ibov", 0)}')
    print(f'Volume relativo >= vol_relativo_min: {_val(row, "vol_relativo")} >= {p["vol_relativo_min"]} -> {_val(row, "vol_relativo") >= p["vol_relativo_min"]}')
    print(f'ADX 14 <= adx_max: {_val(row, "ADX_14")} <= {p["adx_max"]} -> {_val(row, "ADX_14") <= p["adx_max"]}')


    return (
        _val(row, "rsi") >= p["rsi_min"]
        and _val(row, "rsi") <= p["rsi_max"]
        and _retorno_nd(row, 5) <= p["queda_5d_min"]
        and _val(row, "close") > _val(row, "mm50", 0)
        and _val(row, "close_ibov") > _val(row, "mm200_ibov", 0)
        and _val(row, "vol_relativo") >= p["vol_relativo_min"]
        and _val(row, "ADX_14") <= p["adx_max"]
    )

In [12]:
row = df.iloc[-5]
fn_sinal(row, p_mrsl)


RSI >= rsi_min: 37.061573587220124 >= 20 -> True
RSI <= rsi_max: 37.061573587220124 <= 45 -> True
Retorno 5d <= queda_5d_min: -0.14385619391923554 <= -0.02 -> True
Close > MM20: 8.569999694824219 > 9.761500024795533 -> False
Close Ibov > MM200 Ibov: 180915.0 > 151844.935 -> True
Volume relativo >= vol_relativo_min: 2.2064382983196857 >= 0.5 -> True
ADX 14 <= adx_max: 26.115409036508595 <= 40 -> True


False